# YOLO Bukukia Training Pipeline — v3.1 (DPI-Aware)

End-to-end notebook for training with **DPI-aware preprocessing**.
600 DPI images are downsampled to 96 DPI for training; YOLO labels use normalized coordinates (0–1) so they remain valid after resizing.

## Pipeline Overview

| Step | Section | Script |
|---|---|---|
| 0 | Setup Environment | — |
| 1 | Configuration | — |
| 2 | DPI Preprocessing (600→96) | inline |
| 3 | Training (Focal Loss) | `train_2.py` |
| 4 | Model Evaluation | `test_model.py` |
| 5 | Fine-Tuning *(optional)* | `fine_tune.py` |
| 6 | Manual / Advanced Tools | various |

---
## 0. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install ultralytics>=8.3.0 opencv-python-headless matplotlib pandas Pillow -q

# Set working directory
import os, sys
project_path = '/content/drive/MyDrive/Yolo_Autolabel'
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Working dir: {os.getcwd()}")
else:
    print("WARNING: Project path not found!")

# Fix stdout buffer issue in Colab
if not hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure = lambda **kwargs: None

---
## 1. Configuration

Set the model, training parameters, and DPI settings here.

| Parameter | Description | Default |
|---|---|---|
| `YOLO_MODEL` | Model checkpoint name or path | `yolo26n.pt` |
| `YOLO_EPOCHS` | Number of training epochs | `50` |
| `YOLO_CONF` | Confidence threshold | `0.25` |
| `ORIGINAL_DPI` | Resolution of production pages | `600` |
| `TRAINING_DPI` | Resolution for YOLO training | `96` |

In [ ]:
import os

# ── Model Settings ──
os.environ['YOLO_MODEL']  = 'yolo26n.pt'
os.environ['YOLO_EPOCHS'] = '50'
os.environ['YOLO_CONF']   = '0.25'

# ── DPI Settings ──
os.environ['ORIGINAL_DPI'] = '600'
os.environ['TRAINING_DPI'] = '96'

# ── Verify ──
original_dpi = int(os.environ['ORIGINAL_DPI'])
training_dpi = int(os.environ['TRAINING_DPI'])
scale_factor = original_dpi / training_dpi

print(f"Model        : {os.environ['YOLO_MODEL']}")
print(f"Epochs       : {os.environ['YOLO_EPOCHS']}")
print(f"Original DPI : {original_dpi}")
print(f"Training DPI : {training_dpi}")
print(f"Scale Factor : {scale_factor}x")

---
## 2. DPI Preprocessing — Downsample 600 → 96 DPI

Convert 600 DPI production images to 96 DPI for YOLO training.

| | Path |
|---|---|
| **Input** | `input_files/raw_images/*.png` — Original 600 DPI page scans |
| **Output** | `input_files/raw_images_96dpi/*.png` — Downsampled 96 DPI copies |

> ⚠️ The original 600 DPI images are **preserved** in `raw_images/`.
>
> YOLO labels use **normalized coordinates (0–1)**, so they remain valid after resizing — no label changes needed.

In [ ]:
from pathlib import Path
from PIL import Image
import os

original_dpi = int(os.environ.get('ORIGINAL_DPI', 600))
training_dpi = int(os.environ.get('TRAINING_DPI', 96))
scale_factor = original_dpi / training_dpi   # 6.25

RAW_IMAGES = Path('input_files/raw_images')
DOWNSAMPLED_DIR = Path('input_files/raw_images_96dpi')
DOWNSAMPLED_DIR.mkdir(parents=True, exist_ok=True)

extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'}
images = [f for f in RAW_IMAGES.iterdir() if f.suffix.lower() in extensions]

print(f"Found {len(images)} images in {RAW_IMAGES}")
print(f"Downsampling {original_dpi} DPI → {training_dpi} DPI (÷{scale_factor})")
print(f"Output: {DOWNSAMPLED_DIR}")
print("-" * 50)

skipped = 0
processed = 0

for img_path in sorted(images):
    out_path = DOWNSAMPLED_DIR / img_path.name

    # Skip if already downsampled and newer than source
    if out_path.exists() and out_path.stat().st_mtime >= img_path.stat().st_mtime:
        skipped += 1
        continue

    img = Image.open(img_path)
    new_w = int(img.width / scale_factor)
    new_h = int(img.height / scale_factor)
    img_resized = img.resize((new_w, new_h), Image.LANCZOS)
    img_resized.save(out_path)
    processed += 1

    if processed <= 3:
        print(f"  {img_path.name}: {img.width}×{img.height} → {new_w}×{new_h}")

print(f"\nDone! Processed: {processed} | Skipped (cached): {skipped}")

---
## 3. Training — `train_2.py` (Focal Loss)

Trains the YOLO model with **Focal Loss** support and **IoU callback** tracking.
Uses the 96 DPI downsampled images from Step 2.

| | Path / Description |
|---|---|
| **Input** | `input_files/raw_images_96dpi/` — 96 DPI images (from Step 2) |
| **Input** | `input_files/export/labels/*.txt` — YOLO labels (normalized 0–1) |
| **Input** | `input_files/export/classes.txt` — Class names |
| **Output** | `results/runs/train/weights/best.pt` — Best trained model |
| **Output** | `results/runs/train/training_analysis.png` — Training dashboard |
| **Output** | `results/runs/train/iou_log.csv` — Per-epoch Mean IoU log |

### Arguments

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | config | Model name or path |
| `--epochs` | int | config | Number of training epochs |
| `--imgsz` | int | `640` | Image size for training |
| `--batch` | int | `-1` | Batch size (`-1` = auto) |
| `--limit` | int | `0` | Limit total images (0 = use all) |
| `--fraction` | float | `0.0` | Use % of data (overrides `--limit`) |
| `--images-dir` | str | `raw_images` | Custom image folder path |
| `--split-dir` | str | — | Pre-split dataset directory (skips auto-split) |
| `--fl-gamma` | float | `1.5` | Focal Loss gamma (0.0 = disable) |
| `--fl-alpha` | float | `0.25` | Focal Loss alpha (class-balance factor) |

> 💡 Use `--images-dir input_files/raw_images_96dpi` to point to the downsampled images.

In [ ]:
# ── Training with Focal Loss on 96 DPI images ──
# Adjust arguments as needed:

!python scripts/train_2.py \
    --model $YOLO_MODEL \
    --epochs $YOLO_EPOCHS \
    --imgsz 640 \
    --batch -1 \
    --images-dir input_files/raw_images_96dpi \
    --fl-gamma 1.5 \
    --fl-alpha 0.25

### Alternative: Train with Pre-Split Dataset

If you already have a pre-split 96 DPI dataset, use `--split-dir` to skip auto-split.

In [ ]:
# ── Alternative: Train with pre-split dataset ──
# Uncomment and adjust the --split-dir path:

# !python scripts/train_2.py \
#     --model $YOLO_MODEL \
#     --epochs $YOLO_EPOCHS \
#     --imgsz 640 \
#     --batch -1 \
#     --fl-gamma 1.5 \
#     --fl-alpha 0.25 \
#     --split-dir input_files/dataset_fixpage96

---
## 4. Model Evaluation — `test_model.py`

Evaluate the trained model on a separate test set.

| | Path / Description |
|---|---|
| **Input** | `results/runs/train/weights/best.pt` — Trained model |
| **Input** | `input_files/test-dataset/` — Test images (with optional GT labels) |
| **Output** | `results/test_results/test_dashboard.png` — Visual dashboard |
| **Output** | `results/test_results/annotated/*.jpg` — Annotated detection images |
| **Output** | `results/test_results/test_report.txt` — Metrics summary |

### Arguments

| Argument | Type | Default | Description |
|---|---|---|---|
| `--conf` | float | `0.25` | Confidence threshold |
| `--model` | str | auto-detected | Path to model file (.pt) |
| `--data` | str | `test-dataset` | Test images directory |

In [ ]:
# ── Model Evaluation ──

!python scripts/test_model.py \
    --conf  $YOLO_CONF \
    --model results/runs/train/weights/best.pt \
    --data  input_files/test-dataset

---
## 5. Fine-Tuning *(Optional)* — `fine_tune.py`

Ultralytics built-in `model.tune()` for hyperparameter search.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | auto-detected | Model .pt to fine-tune |
| `--iterations` | int | `30` | Tuning iterations |
| `--epochs` | int | `30` | Epochs per trial |
| `--imgsz` | int | `640` | Image size |

> ⏱️ Each iteration runs full training. Expect several hours on a T4 GPU.

In [ ]:
# ── Fine-Tuning (Optional) ──
# Uncomment to run:

# !python scripts/fine_tune.py \
#     --model results/runs/train/weights/best.pt \
#     --iterations 30 \
#     --epochs 30 \
#     --imgsz 640

---
---
# 6. Manual / Advanced Tools

Standalone utility scripts for evaluation, data export, and data management.
Each section below is independent and ready to run.

---
### 6.1 Evaluate IoU — Validation Set (`evaluate_iou_full.py`)

Full IoU evaluation on the validation dataset.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Path to trained model (.pt) |
| `--data` | str | **required** | Validation dataset directory |
| `--out` | str | `.` | Output directory |
| `--conf` | float | `0.25` | Confidence threshold |
| `--iou-thres` | float | `0.5` | IoU threshold for matching |

In [ ]:
!python scripts/evaluate_iou_full.py \
    --model results/runs/train/weights/best.pt \
    --data  results/dataset \
    --out   results/iou_results

---
### 6.2 Evaluate IoU — Training Set (`evaluate_iou_full_train.py`)

Same IoU evaluation on the **training** set — useful for checking overfitting.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Path to trained model (.pt) |
| `--data` | str | **required** | Dataset directory |
| `--out` | str | `.` | Output directory |
| `--conf` | float | `0.25` | Confidence threshold |
| `--iou-thres` | float | `0.5` | IoU threshold for matching |

In [ ]:
!python scripts/evaluate_iou_full_train.py \
    --model results/runs/train/weights/best.pt \
    --data  results/dataset \
    --out   results/iou_results_train

---
### 6.3 Predict to Labels (`predict_to_labels.py`)

Run YOLO predictions and save as YOLO-format label `.txt` files.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Path to trained model (.pt) |
| `--source` | str | **required** | Directory of images to predict on |
| `--out` | str | `predicted_labels` | Output directory for label files |
| `--conf` | float | `0.25` | Confidence threshold |
| `--imgsz` | int | `640` | Image size for inference |

In [ ]:
!python scripts/predict_to_labels.py \
    --model  results/runs/train/weights/best.pt \
    --source input_files/raw_images_96dpi \
    --out    results/predicted_labels \
    --conf   0.25

---
### 6.4 Extract Label Details (`extract_label_details.py`)

Extract label annotations from `.txt` files into a CSV with class names and bounding box info.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Directory of YOLO label .txt files |
| `--classes-txt` | str | **required** | Path to `classes.txt` |
| `--out` | str | `label_details.csv` | Output CSV file path |

In [ ]:
!python scripts/extract_label_details.py \
    --labels-dir  input_files/export/labels \
    --classes-txt input_files/export/classes.txt \
    --out         results/label_details.csv

---
### 6.5 Manual Hyperparameter Tuning (`fine_tune_tuningmanual.py`)

Coordinate-descent sweep with heatmap/scatter visualization.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | auto-detected | Model .pt |
| `--epochs` | int | `30` | Epochs per trial |
| `--imgsz` | int | `640` | Image size |
| `--batch` | int | `16` | Batch size |
| `--rounds` | int | `2` | Coordinate descent rounds |

In [ ]:
# Uncomment to run:

# !python scripts/fine_tune_tuningmanual.py \
#     --model results/runs/train/weights/best.pt \
#     --epochs 30 \
#     --imgsz 640 \
#     --batch 16 \
#     --rounds 2

---
### 6.6 Copy Images from Labels (`copy_images_from_labels_colab.py`)

Copy images whose filenames match existing label filenames.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Directory of label .txt files |
| `--images-dir` | str | **required** | Source images directory |
| `--out` | str | **required** | Destination directory |

In [ ]:
!python scripts/copy_images_from_labels_colab.py \
    --labels-dir input_files/export/labels \
    --images-dir input_files/raw_images_96dpi \
    --out        input_files/matched_images

---
### 6.7 Filter Images by CSV (`filter_images_colab.py`)

Copy images whose filenames appear in a CSV.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--csv` | str | **required** | CSV file with `filename` column |
| `--images-dir` | str | **required** | Source images directory |
| `--out` | str | **required** | Destination directory |

In [ ]:
!python scripts/filter_images_colab.py \
    --csv        results/filter_list.csv \
    --images-dir input_files/raw_images_96dpi \
    --out        input_files/filtered_images

---
### 6.8 Filter Labels by CSV (`filter_labels_colab.py`)

Copy labels whose filenames appear in a CSV.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--csv` | str | **required** | CSV file with `filename` column |
| `--labels-dir` | str | **required** | Source labels directory |
| `--out` | str | **required** | Destination directory |

In [ ]:
!python scripts/filter_labels_colab.py \
    --csv        results/filter_list.csv \
    --labels-dir input_files/export/labels \
    --out        input_files/filtered_labels

---
### 6.9 Filter Labels by Segment Class (`filter_labels_by_segment_colab.py`)

Filter label files to keep only rows matching specific class IDs.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Source labels directory |
| `--out` | str | **required** | Destination directory |
| `--classes` | int[] | **required** | Class IDs to keep (space-separated) |

In [ ]:
!python scripts/filter_labels_by_segment_colab.py \
    --labels-dir input_files/export/labels \
    --out        input_files/filtered_by_segment \
    --classes 0 1 2

---
### 6.10 Move/Copy Random Images (`move_random_images_colab.py`)

Randomly select N images (and matching labels) and copy/move to a target directory.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--images-dir` | str | **required** | Source images directory |
| `--labels-dir` | str | — | Source labels directory |
| `--out` | str | **required** | Destination directory |
| `--count` | int | **required** | Number of images to select |
| `--move` | flag | `False` | Move instead of copy |
| `--seed` | int | `42` | Random seed |

In [ ]:
!python scripts/move_random_images_colab.py \
    --images-dir input_files/raw_images_96dpi \
    --labels-dir input_files/export/labels \
    --out        input_files/test-dataset \
    --count 20 \
    --seed 42